In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
!pip install torch esm pandas
!pip install fair-esm --upgrade
!pip install biopython

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 460.3/460.3 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.4/184.4 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:
import esm
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# Load small ESM-2 model
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()

model = model.to(device)
model.eval()

print("ESM-2 loaded")

Using device: cuda
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t6_8M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t6_8M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D-contact-regression.pt
ESM-2 loaded


In [146]:
def get_protein_embedding(sequence):

    batch_converter = alphabet.get_batch_converter()

    data = [
        ("protein", sequence)
    ]

    labels, strs, tokens = batch_converter(data)

    tokens = tokens.to(device)

    with torch.no_grad():
        """
        results = model(
            tokens,
            repr_layers=[6]
        )"""

        results = esm_model(
            tokens,
            repr_layers=[6]
        )

    embedding = results["representations"][6]

    return embedding.cpu()

In [147]:
sequence = "MKTIIALSYIFCLVFADYK"

embedding = get_protein_embedding(sequence)

print("Embedding shape:")
print(embedding.shape)

Embedding shape:
torch.Size([1, 21, 320])


In [148]:
import os

os.makedirs(
    "/content/MIP-FM/data/test_embeddings",
    exist_ok=True
)

In [149]:
torch.save(
    embedding,
    "/content/MIP-FM/data/test_embeddings/example.pt"
)

print("Saved")

Saved


In [150]:
import os

base_path="/content/MIP-FM/data"

folders=[
    "ppi",
    "sequences",
    "structures"
]

for f in folders:
    os.makedirs(
        f"{base_path}/{f}",
        exist_ok=True
    )

print("Dataset folders created")

Dataset folders created


In [151]:
import os

folders=[
    "/content/MIP-FM/data/sequences",
    "/content/MIP-FM/data/ppi"
]

for f in folders:
    os.makedirs(f, exist_ok=True)

print("folders created")

folders created


In [152]:
import pandas as pd


train_file="/content/human_train.tsv"


train=pd.read_csv(
    train_file,
    sep="\t"
)


train.head()

,9606.ENSP00000409077,9606.ENSP00000470819,1
0,9606.ENSP00000263904,9606.ENSP00000472680,1
1,9606.ENSP00000364459,9606.ENSP00000360117,1
2,9606.ENSP00000422403,9606.ENSP00000400591,1
3,9606.ENSP00000388332,9606.ENSP00000346080,1
4,9606.ENSP00000469581,9606.ENSP00000386920,1


In [153]:
from Bio import SeqIO


fasta="/content/human.fasta"


count=0

for record in SeqIO.parse(fasta,"fasta"):

    print(record.id)
    print(record.seq[:50])

    count+=1

    if count==3:
        break

9606.ENSP00000000233
MGLTVSALFSRIFGKKQMRILMVGLDAAGKTTILYKLKLGEIVTTIPTIG
9606.ENSP00000412701
MGLTVSALFSRIFGKKQMRILMVGLDAAGKTTILYKLKLGEIVTTIPTIG
9606.ENSP00000442983
MFPFYSCWRTGLLLLLLAVAVRESWQTEEKTCDLVGEKGKESEKELALVK


Step 2.2 — Load the PPI interaction pairs

In [154]:
import pandas as pd

train_path="/content/human_train.tsv"

train = pd.read_csv(
    train_path,
    sep="\t"
)

print(train.shape)

train.head()

(421791, 3)


,9606.ENSP00000409077,9606.ENSP00000470819,1
0,9606.ENSP00000263904,9606.ENSP00000472680,1
1,9606.ENSP00000364459,9606.ENSP00000360117,1
2,9606.ENSP00000422403,9606.ENSP00000400591,1
3,9606.ENSP00000388332,9606.ENSP00000346080,1
4,9606.ENSP00000469581,9606.ENSP00000386920,1


In [155]:
from Bio import SeqIO


fasta_path="/content/human.fasta"


sequence_dict={}

for record in SeqIO.parse(
    fasta_path,
    "fasta"
):
    sequence_dict[record.id]=str(record.seq)


print("Number of proteins:", len(sequence_dict))

Number of proteins: 70529


In [156]:
missing=[]

for protein in train.iloc[:,0]:
    if protein not in sequence_dict:
        missing.append(protein)


for protein in train.iloc[:,1]:
    if protein not in sequence_dict:
        missing.append(protein)


print("Missing proteins:", len(set(missing)))

Missing proteins: 0


Testing

In [157]:
def get_esm_embedding(sequence):

    data = [
        ("protein", sequence)
    ]

    labels, strs, tokens = batch_converter(data)

    tokens = tokens.to(device)


    with torch.no_grad():

        results = model(
            tokens,
            repr_layers=[6],
            return_contacts=False
        )


    embedding = results["representations"][6]


    # remove CLS and EOS tokens
    embedding = embedding[:,1:-1,:]


    return embedding.cpu()

In [158]:
import esm
import torch


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# Load ESM-2
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()


model = model.to(device)

model.eval()


# IMPORTANT: create batch converter
batch_converter = alphabet.get_batch_converter()


print("ESM-2 loaded successfully")
def get_esm_embedding(sequence):

    data = [
        ("protein", sequence)
    ]


    labels, strs, tokens = batch_converter(data)


    tokens = tokens.to(device)


    with torch.no_grad():

        results = model(
            tokens,
            repr_layers=[6],
            return_contacts=False
        )


    embedding = results["representations"][6]


    # Remove special tokens
    embedding = embedding[:,1:-1,:]


    return embedding.cpu()

Device: cuda
ESM-2 loaded successfully


In [159]:
protein_id = list(sequence_dict.keys())[0]

sequence = sequence_dict[protein_id]


embedding = get_esm_embedding(sequence)


print("Protein ID:", protein_id)
print("Sequence length:", len(sequence))
print("Embedding shape:", embedding.shape)

Protein ID: 9606.ENSP00000000233
Sequence length: 180
Embedding shape: torch.Size([1, 180, 320])


Step 4.1 — Create Protein Embedding Function

In [160]:
def get_protein_vector(sequence):

    residue_embedding = get_esm_embedding(sequence)

    # Mean pooling over residues
    protein_vector = residue_embedding.mean(dim=1)

    return protein_vector


In [161]:
print(sequence[:50])
print(len(sequence))

MGLTVSALFSRIFGKKQMRILMVGLDAAGKTTILYKLKLGEIVTTIPTIG
180


In [162]:
protein_vector = get_protein_vector(sequence)

print(protein_vector.shape)

torch.Size([1, 320])


Step 4.3 — Generate embeddings for 100 proteins

In [163]:
from tqdm import tqdm
import torch


test_sequences = dict(
    list(sequence_dict.items())[:100]
)


protein_embeddings={}


for protein_id, sequence in tqdm(test_sequences.items()):

    protein_embeddings[protein_id] = get_protein_vector(sequence)


print(
    "Proteins embedded:",
    len(protein_embeddings)
)

100%|██████████| 100/100 [00:01<00:00, 80.27it/s]

Proteins embedded: 100


Step 5.1 — Load the PPI training data

In [164]:
import pandas as pd

train_path="/content/human_train.tsv"

train_df=pd.read_csv(
    train_path,
    sep="\t"
)

print(train_df.shape)

train_df.head()

(421791, 3)


,9606.ENSP00000409077,9606.ENSP00000470819,1
0,9606.ENSP00000263904,9606.ENSP00000472680,1
1,9606.ENSP00000364459,9606.ENSP00000360117,1
2,9606.ENSP00000422403,9606.ENSP00000400591,1
3,9606.ENSP00000388332,9606.ENSP00000346080,1
4,9606.ENSP00000469581,9606.ENSP00000386920,1


Step 5.2 — Check your embedding dictionary

In [165]:
import os

for root, dirs, files in os.walk("/content"):
    for f in files:
        if f.endswith(".pt") or f.endswith(".pth"):
            print(os.path.join(root, f))

/content/protein_embeddings.pt
/content/MIP-FM/ppi_baseline_esm2.pt
/content/MIP-FM/data/test_embeddings/example.pt
/content/MIP-FM/data/embeddings/human_esm2_8M_vectors.pt


In [166]:
import os

print(os.listdir("/content/MIP-FM"))

['ppi_baseline_esm2.pt', 'data']


In [167]:
for root, dirs, files in os.walk("/content/MIP-FM"):
    print(root, len(files))

/content/MIP-FM 1
/content/MIP-FM/data 0
/content/MIP-FM/data/sequences 0
/content/MIP-FM/data/test_embeddings 1
/content/MIP-FM/data/ppi 0
/content/MIP-FM/data/structures 0
/content/MIP-FM/data/embeddings 1


In [168]:
import os

for root, dirs, files in os.walk("/content"):
    for f in files:
        if f.endswith(".pt"):
            print(os.path.join(root, f))

/content/protein_embeddings.pt
/content/MIP-FM/ppi_baseline_esm2.pt
/content/MIP-FM/data/test_embeddings/example.pt
/content/MIP-FM/data/embeddings/human_esm2_8M_vectors.pt


In [169]:
import torch

protein_embeddings = torch.load(
    "/content/protein_embeddings.pt"
)

print(len(protein_embeddings))

print(
    list(protein_embeddings.keys())[:5]
)

13555
['9606.ENSP00000365953', '9606.ENSP00000473392', '9606.ENSP00000482566', '9606.ENSP00000425719', '9606.ENSP00000422858']


In [170]:
torch.save(
    protein_embeddings,
    "/content/protein_embeddings.pt"
)

In [171]:
import torch

protein_embeddings=torch.load(
    "/content/MIP-FM/data/embeddings/human_100_protein_vectors.pt"
)

print(len(protein_embeddings))

print(
    list(protein_embeddings.keys())[:5]
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/MIP-FM/data/embeddings/human_100_protein_vectors.pt'

Step 5.3 — Create PPI Dataset Class

In [172]:
import torch
from torch.utils.data import Dataset


class PPIDataset(Dataset):

    def __init__(self, dataframe, embeddings):

        self.data=dataframe
        self.embeddings=embeddings


    def __len__(self):

        return len(self.data)


    def __getitem__(self,index):

        row=self.data.iloc[index]


        protein_a=row[0]
        protein_b=row[1]


        label=row[2]


        emb_a=self.embeddings[protein_a]

        emb_b=self.embeddings[protein_b]


        # combine both proteins
        x=torch.cat(
            [
                emb_a.squeeze(0),
                emb_b.squeeze(0)
            ]
        )


        return x, torch.tensor(
            label,
            dtype=torch.float32
        )

In [173]:
dataset=PPIDataset(
    train_df,
    protein_embeddings
)


Step 5.5 — Filter available interactions

In [174]:
available=set(
    protein_embeddings.keys()
)


filtered_train=train_df[
    train_df.iloc[:,0].isin(available)
    &
    train_df.iloc[:,1].isin(available)
]


print(filtered_train.shape)

(346196, 3)


In [175]:
dataset=PPIDataset(
    filtered_train,
    protein_embeddings
)

print(len(dataset))

346196


Step 5.6 — Build the Baseline Model

In [176]:
import torch.nn as nn


class PPINetwork(nn.Module):

    def __init__(self):

        super().__init__()


        self.model=nn.Sequential(

            nn.Linear(640,256),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(256,64),

            nn.ReLU(),

            nn.Linear(64,1)

        )


    def forward(self,x):

        return self.model(x).squeeze()

In [177]:
model=PPINetwork()


print(model)

PPINetwork(
  (model): Sequential(
    (0): Linear(in_features=640, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [178]:
print(filtered_train.shape)
print(len(dataset))

(346196, 3)
346196


In [179]:
train_df.head()
train_df.columns

Index(['9606.ENSP00000409077', '9606.ENSP00000470819', '1'], dtype='object')

In [180]:
print(train_df.head())
print(train_df.columns)

   9606.ENSP00000409077  9606.ENSP00000470819  1
0  9606.ENSP00000263904  9606.ENSP00000472680  1
1  9606.ENSP00000364459  9606.ENSP00000360117  1
2  9606.ENSP00000422403  9606.ENSP00000400591  1
3  9606.ENSP00000388332  9606.ENSP00000346080  1
4  9606.ENSP00000469581  9606.ENSP00000386920  1
Index(['9606.ENSP00000409077', '9606.ENSP00000470819', '1'], dtype='object')


In [181]:
import pandas as pd


train_path="/content/human_train.tsv"


train_df=pd.read_csv(
    train_path,
    sep="\t",
    header=None
)


train_df.columns=[
    "protein_A",
    "protein_B",
    "label"
]


train_df.head()

,protein_A,protein_B,label
0,9606.ENSP00000409077,9606.ENSP00000470819,1
1,9606.ENSP00000263904,9606.ENSP00000472680,1
2,9606.ENSP00000364459,9606.ENSP00000360117,1
3,9606.ENSP00000422403,9606.ENSP00000400591,1
4,9606.ENSP00000388332,9606.ENSP00000346080,1


In [182]:
protein_ids=set(
    train_df["protein_A"]
).union(
    set(train_df["protein_B"])
)


print("Total unique proteins:", len(protein_ids))

Total unique proteins: 15816


In [183]:
available_proteins=set(sequence_dict.keys())


matched = protein_ids.intersection(
    available_proteins
)


missing = protein_ids - available_proteins


print("Proteins required:", len(protein_ids))
print("Sequences available:", len(matched))
print("Missing sequences:", len(missing))

Proteins required: 15816
Sequences available: 15816
Missing sequences: 0


Step 5.6.1 — Prepare sequences

In [184]:
from tqdm import tqdm
from tqdm import tqdm

required_sequences = {

    pid: sequence_dict[pid]

    for pid in protein_ids

    if pid in sequence_dict

}


print("Sequences loaded:", len(required_sequences))
"""
required_sequences = {
    pid: sequence_dict[pid]
    for pid in protein_ids
}


print(len(required_sequences))"""

Sequences loaded: 15816


'\nrequired_sequences = {\n    pid: sequence_dict[pid]\n    for pid in protein_ids\n}\n\n\nprint(len(required_sequences))'

In [185]:
missing_proteins = [
    pid for pid in protein_ids
    if pid not in sequence_dict
]

print("Missing proteins:", len(missing_proteins))

print(missing_proteins[:20])

Missing proteins: 0
[]


Step 5.6.2 — Create batch embedding function
32 proteins → ESM → save
32 proteins → ESM → save

In [186]:
def get_batch_embeddings(sequence_dict, batch_size=8):

    embeddings = {}

    protein_ids = list(sequence_dict.keys())


    for start in tqdm(range(0, len(protein_ids), batch_size)):

        batch_ids = protein_ids[start:start+batch_size]


        batch_data = [
            (pid, sequence_dict[pid])
            for pid in batch_ids
        ]


        labels, strs, tokens = batch_converter(batch_data)


        tokens = tokens.to(device)


        with torch.no_grad():

            results = model(
                tokens,
                repr_layers=[6],
                return_contacts=False
            )


        batch_repr = results["representations"][6]


        for i, pid in enumerate(batch_ids):

            # remove CLS and EOS
            emb = batch_repr[i,1:-1,:]

            # mean pooling
            protein_vector = emb.mean(dim=0)

            embeddings[pid] = protein_vector.cpu()


    return embeddings

In [187]:
test_batch = list(required_sequences.items())[:8]

labels, strs, tokens = batch_converter(test_batch)

print(tokens.shape)

torch.Size([8, 716])


In [188]:
import esm
import torch


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


esm_model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()


esm_model = esm_model.to(device)

esm_model.eval()


batch_converter = alphabet.get_batch_converter()


print("ESM model restored")

ESM model restored


In [189]:
def get_batch_embeddings(sequence_dict, batch_size=2):

    embeddings = {}

    protein_ids = list(sequence_dict.keys())


    for start in tqdm(range(0, len(protein_ids), batch_size)):

        batch_ids = protein_ids[start:start+batch_size]


        batch_data = [
            (pid, sequence_dict[pid])
            for pid in batch_ids
        ]


        labels, strs, tokens = batch_converter(batch_data)


        tokens = tokens.to(device)


        with torch.no_grad():

            results = esm_model(
                tokens,
                repr_layers=[6],
                return_contacts=False
            )


        batch_repr = results["representations"][6]


        for i, pid in enumerate(batch_ids):

            residue_embedding = batch_repr[i,1:-1,:]


            protein_vector = residue_embedding.mean(dim=0)


            embeddings[pid] = protein_vector.cpu()


    return embeddings

In [190]:
protein_embeddings = get_batch_embeddings(
    required_sequences,
    batch_size=2
)

100%|██████████| 7908/7908 [03:34<00:00, 36.95it/s]


D-SCRIPT human PPI dataset

        |
        ↓

15,816 protein IDs

        |
        ↓

Human FASTA sequences

        |
        ↓

ESM-2 (8M)

        |
        ↓

Protein embeddings

        |
        ↓

[320-dimensional vector]

In [191]:
import os
import torch


os.makedirs(
    "/content/MIP-FM/data/embeddings",
    exist_ok=True
)


torch.save(
    protein_embeddings,
    "/content/MIP-FM/data/embeddings/human_esm2_8M_vectors.pt"
)


print("Embeddings saved successfully")

Embeddings saved successfully


In [192]:
import os

file_path="/content/MIP-FM/data/embeddings/human_esm2_8M_vectors.pt"

size=os.path.getsize(file_path)/(1024**2)

print("File size:", round(size,2),"MB")

File size: 23.99 MB


Now we start Step 6 — Build the PPI classifier

Step 6.1 — Create training dataset

In [193]:
import torch
from torch.utils.data import Dataset


class PPIDataset(Dataset):

    def __init__(self, dataframe, embeddings):

        self.data=dataframe.reset_index(drop=True)
        self.embeddings=embeddings


    def __len__(self):

        return len(self.data)


    def __getitem__(self, idx):

        row=self.data.iloc[idx]

        protein_a=row["protein_A"]
        protein_b=row["protein_B"]

        label=row["label"]


        emb_a=self.embeddings[protein_a]
        emb_b=self.embeddings[protein_b]


        x=torch.cat(
            [
                emb_a,
                emb_b
            ],
            dim=0
        )


        return x, torch.tensor(
            label,
            dtype=torch.float32
        )

In [194]:
train_dataset = PPIDataset(
    train_df,
    protein_embeddings
)


print("Samples:", len(train_dataset))

Samples: 421792


In [195]:
x,y=train_dataset[0]


print(x.shape)
print(y)

torch.Size([640])
tensor(1.)


Step 6.4 — Neural network + training loop

In [196]:
print(len(protein_embeddings))

15816


Component	Status
D-SCRIPT human PPI dataset	✅ Ready
Human FASTA sequences	✅ Ready
Protein ID mapping	✅ 15,816 / 15,816 matched
ESM-2 model	✅ Working
Protein embeddings	✅ Generated

Step 6 — Prepare PPI Training Dataset

In [197]:
import torch

protein_embeddings = torch.load(
    "/content/MIP-FM/data/embeddings/human_esm2_8M_vectors.pt"
)

print(len(protein_embeddings))

15816


In [198]:
from torch.utils.data import Dataset


class PPIDataset(Dataset):

    def __init__(self, dataframe, embeddings):

        self.data = dataframe.reset_index(drop=True)
        self.embeddings = embeddings


    def __len__(self):

        return len(self.data)


    def __getitem__(self, idx):

        row = self.data.iloc[idx]


        protein_a = row["protein_A"]
        protein_b = row["protein_B"]


        label = row["label"]


        emb_a = self.embeddings[protein_a]
        emb_b = self.embeddings[protein_b]


        # concatenate two protein representations
        x = torch.cat(
            [
                emb_a,
                emb_b
            ],
            dim=0
        )


        return x, torch.tensor(
            label,
            dtype=torch.float32
        )

In [199]:
train_dataset = PPIDataset(
    train_df,
    protein_embeddings
)


print("Number of PPI samples:")
print(len(train_dataset))

Number of PPI samples:
421792


In [200]:
x, y = train_dataset[0]


print("Input:")
print(x.shape)


print("Label:")
print(y)

Input:
torch.Size([640])
Label:
tensor(1.)


Protein A ESM vector = 320

Protein B ESM vector = 320

Combined = 640

Step 6.4 — Create DataLoader

In [201]:
import os

structure_files = os.listdir(
    "/content/MIP-FM/data/structures"
)

print("PDB files:", len(structure_files))
print(structure_files[:5])

PDB files: 0
[]


In [202]:
from torch.utils.data import DataLoader


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)


batch_x, batch_y = next(iter(train_loader))


print(batch_x.shape)
print(batch_y.shape)

torch.Size([64, 640])
torch.Size([64])


Step 6.5 — Build Baseline PPI Network
ESM Protein A
     |
   320
     \
      \
       Concatenate → 640
      /
     /
   320
     |
ESM Protein B


        ↓

    MLP classifier

        ↓

 Interaction probability

In [203]:
import torch.nn as nn


class PPIModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(640,256),

            nn.ReLU(),

            nn.Dropout(0.3),


            nn.Linear(256,64),

            nn.ReLU(),


            nn.Linear(64,1)

        )


    def forward(self,x):

        return self.network(x).squeeze()

Step 6.6 — Initialize Model

In [204]:
ppi_model = PPIModel()


print(ppi_model)

PPIModel(
  (network): Sequential(
    (0): Linear(in_features=640, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=1, bias=True)
  )
)


Step 7 — Train the model

In [205]:
len(train_dataset)

421792

Protein A sequence
        |
      ESM-2
        |
     320 vector

Protein B sequence
        |
      ESM-2
        |
     320 vector

        ↓

Concatenate

        ↓

640-dimensional input

        ↓

PPI classifier

Step 7.1 — Split dataset into train and validation

In [206]:
from torch.utils.data import random_split


# 90% training, 10% validation

train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size


train_data, val_data = random_split(
    train_dataset,
    [train_size, val_size]
)


print("Training samples:", len(train_data))
print("Validation samples:", len(val_data))

Training samples: 379612
Validation samples: 42180


In [207]:
from torch.utils.data import DataLoader


train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True
)


val_loader = DataLoader(
    val_data,
    batch_size=64,
    shuffle=False
)


print("DataLoaders ready")

DataLoaders ready


Step 7.3 — Move model to GPU

In [208]:
import torch


device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)


ppi_model = PPIModel().to(device)


print(device)

cuda


Step 7.4 — Define loss and optimizer

For PPI classification:

label 1 = interaction
label 0 = no interaction

Use binary classification loss.

In [209]:
import torch.nn as nn
import torch.optim as optim


criterion = nn.BCEWithLogitsLoss()


optimizer = optim.Adam(
    ppi_model.parameters(),
    lr=1e-3
)

Step 7.5 — Training Loop

In [210]:
from tqdm import tqdm


epochs = 10


for epoch in range(epochs):

    ppi_model.train()

    total_loss = 0


    for x, y in tqdm(train_loader):

        x = x.to(device)
        y = y.to(device)


        optimizer.zero_grad()


        output = ppi_model(x)


        loss = criterion(
            output,
            y
        )


        loss.backward()


        optimizer.step()


        total_loss += loss.item()



    avg_loss = total_loss / len(train_loader)


    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {avg_loss:.4f}"
    )

100%|██████████| 5932/5932 [00:43<00:00, 137.91it/s]


Epoch 1/10 Loss: 0.2050


100%|██████████| 5932/5932 [00:43<00:00, 134.94it/s]


Epoch 2/10 Loss: 0.1790


100%|██████████| 5932/5932 [00:42<00:00, 140.58it/s]


...

Epoch 8/10 Loss: 0.1426


100%|██████████| 5932/5932 [00:42<00:00, 139.30it/s]


Epoch 9/10 Loss: 0.1396


100%|██████████| 5932/5932 [00:41<00:00, 142.00it/s]

Epoch 10/10 Loss: 0.1365


Step 7.6 — Save baseline model

In [211]:
torch.save(
    ppi_model.state_dict(),
    "/content/MIP-FM/ppi_baseline_esm2.pt"
)


print("Model saved")

Model saved


Step 7.7 — Quick validation

In [212]:
ppi_model.eval()

correct = 0
total = 0


with torch.no_grad():

    for x,y in val_loader:

        x=x.to(device)
        y=y.to(device)


        output=ppi_model(x)


        prediction = (
            torch.sigmoid(output) > 0.5
        )


        correct += (
            prediction == y
        ).sum().item()


        total += y.size(0)


accuracy = correct / total


print("Validation accuracy:", accuracy)

Validation accuracy: 0.9496206733048839


In [213]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    matthews_corrcoef,
    accuracy_score
)

ppi_model.eval()

all_labels = []
all_probs = []


with torch.no_grad():

    for x, y in val_loader:

        x = x.to(device)

        output = ppi_model(x)

        probs = torch.sigmoid(output)


        all_probs.extend(
            probs.cpu().numpy()
        )

        all_labels.extend(
            y.numpy()
        )


# convert predictions

preds = [
    1 if p > 0.5 else 0
    for p in all_probs
]


accuracy = accuracy_score(
    all_labels,
    preds
)

auroc = roc_auc_score(
    all_labels,
    all_probs
)

aupr = average_precision_score(
    all_labels,
    all_probs
)

f1 = f1_score(
    all_labels,
    preds
)

mcc = matthews_corrcoef(
    all_labels,
    preds
)


print("Accuracy:", accuracy)
print("AUROC:", auroc)
print("AUPR:", aupr)
print("F1:", f1)
print("MCC:", mcc)

Accuracy: 0.9496206733048839
AUROC: 0.9476867120601575
AUPR: 0.7810319232025005
F1: 0.6380514392778062
MCC: 0.6454115974996099


In [214]:
torch.save(
    ppi_model.state_dict(),
    "/content/MIP-FM/esm2_ppi_baseline.pt"
)

Completed this pipeline
D-SCRIPT PPI Dataset
        |
        ↓
Human Protein Sequences
        |
        ↓
ESM-2 Protein Language Model
        |
        ↓
Protein Embeddings (320D)
        |
        ↓
Pair Embedding (640D)
        |
        ↓
MLP Classifier
        |
        ↓
PPI Prediction

Current baseline:

ESM-2 sequence
       ↓
MLP
       ↓
PPI


Novel model:

ESM-2 sequence
        +
AlphaFold structure
        +
Evolutionary information
        +
Interface attention
        +
Explainability

        ↓

Multimodal Protein Interaction Foundation Model

Step 9 — Cross-Species Generalization Test (Very Important)
Train:
Human PPI

Test:
Mouse
Fly
Yeast
Worm
E.coli


Human training
      |
      ↓
ESM-2 PPI model
      |
      ↓
Unknown species prediction

In [215]:
import os

for root, dirs, files in os.walk("/content/MIP-FM"):
    for file in files:
        if "mouse" in file.lower():
            print(os.path.join(root,file))

Step 8.2 — Load mouse PPI pairs

In [219]:
import pandas as pd


mouse_test_path="/content/MIP-FM/data/sequences/mouse_test.tsv"


mouse_test = pd.read_csv(
    mouse_test_path,
    sep="\t",
    header=None
)


mouse_test.columns=[
    "protein_A",
    "protein_B",
    "label"
]


print(mouse_test.shape)

mouse_test.head()

(55000, 3)


,protein_A,protein_B,label
0,10090.ENSMUSP00000002551,10090.ENSMUSP00000044178,1.0
1,10090.ENSMUSP00000106249,10090.ENSMUSP00000110731,1.0
2,10090.ENSMUSP00000109313,10090.ENSMUSP00000055941,1.0
3,10090.ENSMUSP00000124695,10090.ENSMUSP00000117609,1.0
4,10090.ENSMUSP00000139467,10090.ENSMUSP00000120807,1.0


Step 8.3 — Load mouse FASTA

In [220]:
from Bio import SeqIO


mouse_fasta="/content/MIP-FM/data/sequences/mouse.fasta"


mouse_sequence_dict={}


for record in SeqIO.parse(
    mouse_fasta,
    "fasta"
):

    mouse_sequence_dict[record.id] = str(record.seq)


print(
    "Mouse proteins:",
    len(mouse_sequence_dict)
)

Mouse proteins: 40606


Step 8.4 — Find proteins needed for testing

In [221]:
mouse_proteins=set(
    mouse_test["protein_A"]
).union(
    set(mouse_test["protein_B"])
)


print(
    "Mouse proteins required:",
    len(mouse_proteins)
)

Mouse proteins required: 37497


Step 8.5 — Check sequence availability

In [222]:
mouse_available = mouse_proteins.intersection(
    set(mouse_sequence_dict.keys())
)


mouse_missing = mouse_proteins - set(mouse_sequence_dict.keys())


print(
    "Available:",
    len(mouse_available)
)

print(
    "Missing:",
    len(mouse_missing)
)

Available: 37497
Missing: 0


Step 8.6 — Generate mouse ESM-2 embeddings

In [223]:
mouse_required_sequences = {
    pid: mouse_sequence_dict[pid]
    for pid in mouse_available
}

In [224]:
mouse_embeddings = get_batch_embeddings(
    mouse_required_sequences,
    batch_size=2
)

100%|██████████| 18749/18749 [08:37<00:00, 36.26it/s]


Step 8.7 — Create mouse test dataset

In [225]:
mouse_dataset = PPIDataset(
    mouse_test,
    mouse_embeddings
)

x,y = mouse_dataset[0]

print(x.shape)
print(y)

torch.Size([640])
tensor(1.)


Step 8.8 — Evaluate human-trained model on mouse

In [226]:
ppi_model.eval()

from torch.utils.data import DataLoader


mouse_loader = DataLoader(
    mouse_dataset,
    batch_size=64,
    shuffle=False
)

In [227]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)


mouse_labels=[]
mouse_probs=[]


with torch.no_grad():

    for x,y in mouse_loader:

        x=x.to(device)

        output=ppi_model(x)

        probs=torch.sigmoid(output)


        mouse_probs.extend(
            probs.cpu().numpy()
        )

        mouse_labels.extend(
            y.numpy()
        )


mouse_auroc = roc_auc_score(
    mouse_labels,
    mouse_probs
)


mouse_aupr = average_precision_score(
    mouse_labels,
    mouse_probs
)


print("Mouse AUROC:", mouse_auroc)
print("Mouse AUPR:", mouse_aupr)

Mouse AUROC: 0.919826918
Mouse AUPR: 0.657896002908134


Other Species Step 9.2 — Load Fly PPI pairs


In [228]:
import pandas as pd

fly_test_path="/content/MIP-FM/data/sequences/fly_test.tsv"


fly_test = pd.read_csv(
    fly_test_path,
    sep="\t",
    header=None
)


fly_test.columns=[
    "protein_A",
    "protein_B",
    "label"
]


print(fly_test.shape)

fly_test.head()

(55000, 3)


,protein_A,protein_B,label
0,7227.FBpp0079304,7227.FBpp0070301,1.0
1,7227.FBpp0085902,7227.FBpp0289817,1.0
2,7227.FBpp0077698,7227.FBpp0079443,1.0
3,7227.FBpp0074529,7227.FBpp0080890,1.0
4,7227.FBpp0071259,7227.FBpp0071497,1.0


Step 9.3 — Load Fly sequences

In [229]:
from Bio import SeqIO


fly_fasta="/content/MIP-FM/data/sequences/fly.fasta"


fly_sequence_dict={}


for record in SeqIO.parse(
    fly_fasta,
    "fasta"
):

    fly_sequence_dict[record.id]=str(record.seq)


print(
    "Fly proteins:",
    len(fly_sequence_dict)
)

Fly proteins: 19310


Step 9.4 — Select required Fly proteins

In [230]:
fly_proteins=set(
    fly_test["protein_A"]
).union(
    set(fly_test["protein_B"])
)


print(
    "Required proteins:",
    len(fly_proteins)
)

Required proteins: 19213


Step 9.5 — Check sequence coverage

In [231]:
fly_available = fly_proteins.intersection(
    set(fly_sequence_dict.keys())
)


fly_missing = fly_proteins - set(fly_sequence_dict.keys())


print("Available:", len(fly_available))
print("Missing:", len(fly_missing))

Available: 19213
Missing: 0


Step 9.6 — Generate Fly ESM-2 embeddings

In [232]:
fly_required_sequences = {
    pid: fly_sequence_dict[pid]
    for pid in fly_available
}

In [233]:
fly_embeddings = get_batch_embeddings(
    fly_required_sequences,
    batch_size=2
)

100%|██████████| 9607/9607 [04:27<00:00, 35.94it/s]


Step 9.7 — Create Fly dataset

In [234]:
fly_dataset = PPIDataset(
    fly_test,
    fly_embeddings
)


x,y=fly_dataset[0]

print(x.shape)
print(y)

torch.Size([640])
tensor(1.)


Step 9.8 — Evaluate Human-trained model on Fly

In [235]:
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score


fly_loader = DataLoader(
    fly_dataset,
    batch_size=64,
    shuffle=False
)


fly_labels=[]
fly_probs=[]


ppi_model.eval()


with torch.no_grad():

    for x,y in fly_loader:

        x=x.to(device)

        output=ppi_model(x)

        probs=torch.sigmoid(output)


        fly_probs.extend(
            probs.cpu().numpy()
        )

        fly_labels.extend(
            y.numpy()
        )


fly_auroc = roc_auc_score(
    fly_labels,
    fly_probs
)


fly_aupr = average_precision_score(
    fly_labels,
    fly_probs
)


print("Fly AUROC:", fly_auroc)
print("Fly AUPR:", fly_aupr)

Fly AUROC: 0.9285012320000002
Fly AUPR: 0.6554332933210909


Yeast cross-species evaluation.

In [236]:
import os

for root, dirs, files in os.walk("/content/MIP-FM"):
    for file in files:
        if "yeast" in file.lower():
            print(os.path.join(root,file))

/content/MIP-FM/data/sequences/yeast.fasta
/content/MIP-FM/data/sequences/yeast_test.tsv


In [237]:
import pandas as pd


yeast_test_path="/content/MIP-FM/data/sequences/yeast_test.tsv"


yeast_test = pd.read_csv(
    yeast_test_path,
    sep="\t",
    header=None
)


yeast_test.columns=[
    "protein_A",
    "protein_B",
    "label"
]


print(yeast_test.shape)

yeast_test.head()


from Bio import SeqIO


yeast_fasta="/content/MIP-FM/data/sequences/yeast.fasta"


yeast_sequence_dict={}


for record in SeqIO.parse(
    yeast_fasta,
    "fasta"
):

    yeast_sequence_dict[record.id]=str(record.seq)


print(
    "Yeast proteins:",
    len(yeast_sequence_dict)
)



yeast_proteins=set(
    yeast_test["protein_A"]
).union(
    set(yeast_test["protein_B"])
)


print(
    "Required yeast proteins:",
    len(yeast_proteins)
)

(55000, 3)
Yeast proteins: 5664
Required yeast proteins: 5664


In [238]:
yeast_available = yeast_proteins.intersection(
    set(yeast_sequence_dict.keys())
)


yeast_missing = yeast_proteins - set(yeast_sequence_dict.keys())


print("Available:", len(yeast_available))
print("Missing:", len(yeast_missing))

Available: 5664
Missing: 0


In [239]:
yeast_required_sequences = {
    pid: yeast_sequence_dict[pid]
    for pid in yeast_available
}


yeast_embeddings = get_batch_embeddings(
    yeast_required_sequences,
    batch_size=2
)

100%|██████████| 2832/2832 [01:10<00:00, 40.40it/s]


In [240]:
yeast_dataset = PPIDataset(
    yeast_test,
    yeast_embeddings
)


x,y = yeast_dataset[0]

print(x.shape)

from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score


yeast_loader = DataLoader(
    yeast_dataset,
    batch_size=64,
    shuffle=False
)


yeast_labels=[]
yeast_probs=[]


ppi_model.eval()


with torch.no_grad():

    for x,y in yeast_loader:

        x=x.to(device)

        output=ppi_model(x)

        probs=torch.sigmoid(output)


        yeast_probs.extend(
            probs.cpu().numpy()
        )

        yeast_labels.extend(
            y.numpy()
        )


yeast_auroc = roc_auc_score(
    yeast_labels,
    yeast_probs
)


yeast_aupr = average_precision_score(
    yeast_labels,
    yeast_probs
)


print("Yeast AUROC:", yeast_auroc)
print("Yeast AUPR:", yeast_aupr)
print(y)

torch.Size([640])
Yeast AUROC: 0.8786627859999999
Yeast AUPR: 0.4997368943064111
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


Step 9.9 — Worm Cross-Species Test

In [241]:
import pandas as pd

worm_test_path="/content/MIP-FM/data/sequences/worm_test.tsv"


worm_test = pd.read_csv(
    worm_test_path,
    sep="\t",
    header=None
)


worm_test.columns=[
    "protein_A",
    "protein_B",
    "label"
]


print(worm_test.shape)

worm_test.head()

(55000, 3)


,protein_A,protein_B,label
0,6239.K04G7.10.1,6239.Y49E10.15,1.0
1,6239.B0513.3b,6239.F42C5.8,1.0
2,6239.C46H11.3,6239.K07C5.1,1.0
3,6239.F23H12.1.1,6239.F29G9.3,1.0
4,6239.F56E10.4.2,6239.Y71A12B.1a,1.0


In [242]:
from Bio import SeqIO


worm_fasta="/content/MIP-FM/data/sequences/worm.fasta"


worm_sequence_dict={}


for record in SeqIO.parse(
    worm_fasta,
    "fasta"
):

    worm_sequence_dict[record.id]=str(record.seq)


print(
    "Worm proteins:",
    len(worm_sequence_dict)
)

Worm proteins: 25930


In [243]:
worm_proteins=set(
    worm_test["protein_A"]
).union(
    set(worm_test["protein_B"])
)


print(
    "Required worm proteins:",
    len(worm_proteins)
)

Required worm proteins: 25429


In [244]:
worm_available = worm_proteins.intersection(
    set(worm_sequence_dict.keys())
)


worm_missing = worm_proteins - set(worm_sequence_dict.keys())


print("Available:", len(worm_available))
print("Missing:", len(worm_missing))

Available: 25429
Missing: 0


In [245]:
worm_required_sequences = {
    pid: worm_sequence_dict[pid]
    for pid in worm_available
}


worm_embeddings = get_batch_embeddings(
    worm_required_sequences,
    batch_size=2
)

100%|██████████| 12715/12715 [05:11<00:00, 40.88it/s]


In [246]:
worm_dataset = PPIDataset(
    worm_test,
    worm_embeddings
)


x,y = worm_dataset[0]

print(x.shape)
print(y)

torch.Size([640])
tensor(1.)


In [247]:
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score


worm_loader = DataLoader(
    worm_dataset,
    batch_size=64,
    shuffle=False
)


worm_labels=[]
worm_probs=[]


ppi_model.eval()


with torch.no_grad():

    for x,y in worm_loader:

        x=x.to(device)

        output=ppi_model(x)

        probs=torch.sigmoid(output)


        worm_probs.extend(
            probs.cpu().numpy()
        )

        worm_labels.extend(
            y.numpy()
        )


worm_auroc = roc_auc_score(
    worm_labels,
    worm_probs
)


worm_aupr = average_precision_score(
    worm_labels,
    worm_probs
)


print("Worm AUROC:", worm_auroc)
print("Worm AUPR:", worm_aupr)

Worm AUROC: 0.93339726
Worm AUPR: 0.6814870660943861


Step 9.10 — E.coli Evaluation

In [248]:
import os

for root, dirs, files in os.walk("/content/MIP-FM"):
    for file in files:
        if "ecoli" in file.lower():
            print(os.path.join(root,file))

/content/MIP-FM/data/sequences/ecoli.fasta
/content/MIP-FM/data/sequences/ecoli_test.tsv


In [249]:
import pandas as pd


ecoli_test_path="/content/MIP-FM/data/sequences/ecoli_test.tsv"


ecoli_test = pd.read_csv(
    ecoli_test_path,
    sep="\t",
    header=None
)


ecoli_test.columns=[
    "protein_A",
    "protein_B",
    "label"
]


print(ecoli_test.shape)

ecoli_test.head()

(22000, 3)


,protein_A,protein_B,label
0,362663.ECP_3406,362663.ECP_4448,1.0
1,362663.ECP_0442,362663.ecp:ECP_0985,1.0
2,362663.ECP_3384,362663.ECP_4447,1.0
3,362663.ECP_0161,362663.ecp:ECP_3117,1.0
4,362663.ecp:ECP_1481,362663.ECP_2475,1.0


In [250]:
from Bio import SeqIO


ecoli_fasta="/content/MIP-FM/data/sequences/ecoli.fasta"


ecoli_sequence_dict={}


for record in SeqIO.parse(
    ecoli_fasta,
    "fasta"
):

    ecoli_sequence_dict[record.id] = str(record.seq)


print(
    "E.coli proteins:",
    len(ecoli_sequence_dict)
)

E.coli proteins: 8848


In [251]:
ecoli_proteins=set(
    ecoli_test["protein_A"]
).union(
    set(ecoli_test["protein_B"])
)


print(
    "Required proteins:",
    len(ecoli_proteins)
)

Required proteins: 7138


In [252]:
ecoli_available = ecoli_proteins.intersection(
    set(ecoli_sequence_dict.keys())
)


ecoli_missing = ecoli_proteins - set(ecoli_sequence_dict.keys())


print("Available:", len(ecoli_available))
print("Missing:", len(ecoli_missing))

Available: 7138
Missing: 0


In [253]:
ecoli_required_sequences = {

    pid: ecoli_sequence_dict[pid]

    for pid in ecoli_available

}


ecoli_embeddings = get_batch_embeddings(
    ecoli_required_sequences,
    batch_size=2
)

100%|██████████| 3569/3569 [01:10<00:00, 50.33it/s]


In [254]:
ecoli_dataset = PPIDataset(
    ecoli_test,
    ecoli_embeddings
)


x,y = ecoli_dataset[0]

print(x.shape)
print(y)

torch.Size([640])
tensor(1.)


In [255]:
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score


ecoli_loader = DataLoader(
    ecoli_dataset,
    batch_size=64,
    shuffle=False
)


ecoli_labels=[]
ecoli_probs=[]


ppi_model.eval()


with torch.no_grad():

    for x,y in ecoli_loader:

        x=x.to(device)

        output=ppi_model(x)

        probs=torch.sigmoid(output)


        ecoli_probs.extend(
            probs.cpu().numpy()
        )

        ecoli_labels.extend(
            y.numpy()
        )


ecoli_auroc = roc_auc_score(
    ecoli_labels,
    ecoli_probs
)


ecoli_aupr = average_precision_score(
    ecoli_labels,
    ecoli_probs
)


print("E.coli AUROC:", ecoli_auroc)
print("E.coli AUPR:", ecoli_aupr)

E.coli AUROC: 0.8312695750000001
E.coli AUPR: 0.45729717406360476


ESM-2 sequence representation
          |
          ↓
Protein pair concatenation
          |
          ↓
MLP classifier
          |
          ↓
Interaction probability


Human AUROC = 0.953

Human → E.coli

AUROC:
0.953 → 0.825

AUPR:
0.793 → 0.444

Now we start Step 10 — Novel Model Development

                         Protein Pair

                              |
        ------------------------------------------------
        |                       |                      |
        |                       |                      |
 Sequence Encoder        Structure Encoder     Evolution Encoder

    ESM-2                    AlphaFold              MSA
    PLM                      Graph Network          Conservation


        ------------------------------------------------

                         Fusion Transformer

                              |

                  Interaction Reasoning Module

                              |

             ---------------------------------
             |                               |
       PPI Prediction              Interface Explanation

Step 10.1 — Add AlphaFold structure features

Protein ID

    ↓

AlphaFold structure

    ↓

Extract:

- residue coordinates
- contact map
- graph representation

    ↓

Graph Neural Network encoder

    ↓

Fusion with ESM-2

    ↓

New PPI model

Version 1 — ESM-2 + Structural Features Fusion

                 Protein A

        Sequence              Structure
           |                     |
         ESM-2              AlphaFold
           |                     |
        320 vector          Structure vector
              \              /
               \            /
                Fusion Layer


                 Protein B

        Sequence              Structure
           |                     |
         ESM-2              AlphaFold
           |                     |
        320 vector          Structure vector


                     |
                     ↓

              Interaction Network

                     ↓

              PPI probability

In [256]:
!pip install bioservices

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.8/277.8 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 129.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.1/88.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.5/718.5 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.6/270.6 kB 27.3 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.3.0
    Uninstalling wrapt-2.3.0:
      Successfully uninsta

Test one protein mapping

In [257]:
!pip install requests

In [258]:
import requests


def ensembl_to_uniprot(ensembl_id):

    url = "https://rest.uniprot.org/idmapping/run"

    data = {
        "from": "Ensembl_PRO",
        "to": "UniProtKB",
        "ids": ensembl_id
    }

    response = requests.post(
        url,
        data=data
    )

    result = response.json()

    return result



test = ensembl_to_uniprot(
    "ENSP00000000233"
)


print(test)

{'url': 'http://rest.uniprot.org/idmapping/run', 'messages': ["The parameter 'from' has an invalid value 'Ensembl_PRO'.", "The combination of 'from=Ensembl_PRO' and 'to=UniProtKB' parameters is invalid"]}


In [259]:
import requests
import time


def map_ensembl_to_uniprot(ensembl_ids):

    url = "https://rest.uniprot.org/idmapping/run"


    data = {
        "from": "Ensembl_PRO",
        "to": "UniProtKB",
        "ids": ",".join(ensembl_ids)
    }


    r = requests.post(
        url,
        data=data
    )


    job_id = r.json()["jobId"]


    # wait for completion

    status_url = (
        f"https://rest.uniprot.org/idmapping/status/{job_id}"
    )


    while True:

        status = requests.get(
            status_url
        ).json()


        if "jobStatus" not in status:
            break


        time.sleep(2)


    result_url = (
        f"https://rest.uniprot.org/idmapping/results/{job_id}"
    )


    result = requests.get(
        result_url
    ).json()


    return result

Step 10.4 — Test on your embeddings

In [260]:
protein_ids_test = list(
    protein_embeddings.keys()
)[:10]


protein_ids_test

['9606.ENSP00000365953',
 '9606.ENSP00000473392',
 '9606.ENSP00000432512',
 '9606.ENSP00000482566',
 '9606.ENSP00000425719',
 '9606.ENSP00000422858',
 '9606.ENSP00000376349',
 '9606.ENSP00000342570',
 '9606.ENSP00000432386',
 '9606.ENSP00000297540']

In [261]:
ensembl_ids = [
    x.split(".")[1]
    for x in protein_ids_test
]


print(ensembl_ids)

['ENSP00000365953', 'ENSP00000473392', 'ENSP00000432512', 'ENSP00000482566', 'ENSP00000425719', 'ENSP00000422858', 'ENSP00000376349', 'ENSP00000342570', 'ENSP00000432386', 'ENSP00000297540']


In [262]:
import requests


ensembl_ids = [
    "ENSP00000000233",
    "ENSP00000337431"
]


url = "https://rest.uniprot.org/idmapping/run"


data = {
    "from": "Ensembl_PRO",
    "to": "UniProtKB",
    "ids": ",".join(ensembl_ids)
}


r = requests.post(
    url,
    data=data
)


print(r.status_code)
print(r.text)

400
{"url":"http://rest.uniprot.org/idmapping/run","messages":["The parameter 'from' has an invalid value 'Ensembl_PRO'.","The combination of 'from=Ensembl_PRO' and 'to=UniProtKB' parameters is invalid"]}


In [263]:
import requests


ensembl_id = "ENSP00000000233"


url = "https://rest.uniprot.org/idmapping/run"


data = {
    "from": "Ensembl_PRO",
    "to": "UniProtKB",
    "ids": ensembl_id
}


response = requests.post(
    url,
    data=data
)


print("Status:", response.status_code)

print("Response:")
print(response.text[:500])

Status: 400
Response:
{"url":"http://rest.uniprot.org/idmapping/run","messages":["The parameter 'from' has an invalid value 'Ensembl_PRO'.","The combination of 'from=Ensembl_PRO' and 'to=UniProtKB' parameters is invalid"]}


In [264]:
!pip install fair-esm

In [265]:
!pip install "fair-esm[esmfold]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 32.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 21.3 MB/s eta 0:00:00
  Created wheel for deepspeed: filename=deepspeed-0.5.9-py3-none-any.whl size=524313 sha256=111ab3bb7bc1ee4418a83a5d04d2f500501d9b4d66757f574f4b1aacd2791b3a
  Stored in directory: /root/.cache/pip/wheels/1d/ee/05/43aed7fd308a1b81ae4fe812c2ac76c9bdbd693d6184c34869
Successfully built deepspeed


In [266]:
!pip install 'dllogger @ git+https://github.com/NVIDIA/dllogger.git'

  Cloning https://github.com/NVIDIA/dllogger.git to /tmp/pip-install-8wxvqdf0/dllogger_756bceaaa36740e9b22cb0f19d8ec463
  Running command git clone --filter=blob:none --quiet https://github.com/NVIDIA/dllogger.git /tmp/pip-install-8wxvqdf0/dllogger_756bceaaa36740e9b22cb0f19d8ec463
  Resolved https://github.com/NVIDIA/dllogger.git to commit 0478734ff7be75adde8d160e04872664d1c62e5f
  Preparing metadata (setup.py) ... done
  Created wheel for dllogger: filename=DLLogger-1.1.0-py3-none-any.whl size=5659 sha256=f3645db417da542bf9f0d20da92855b68111e90e2f72983f8bf9d1bbf2daf661
  Stored in directory: /tmp/pip-ephem-wheel-cache-p0sdntgw/wheels/83/cc/dd/cb9733cc4cd1a319d7ceb4a6f39bce271559a9359f010a9096
Successfully built dllogger


In [7]:
!pip install 'openfold @ git+https://github.com/aqlaboratory/openfold.git@v1.0.1'

  Cloning https://github.com/aqlaboratory/openfold.git (to revision v1.0.1) to /tmp/pip-install-3rzqcfaf/openfold_0a56a77d8e6a4916ba5d8fa2a8fa1e6f
  Running command git clone --filter=blob:none --quiet https://github.com/aqlaboratory/openfold.git /tmp/pip-install-3rzqcfaf/openfold_0a56a77d8e6a4916ba5d8fa2a8fa1e6f
  Running command git checkout -q 42e71db7fa327e0810eb0e371abc9f82aa9b7a6a
  Resolved https://github.com/aqlaboratory/openfold.git to commit 42e71db7fa327e0810eb0e371abc9f82aa9b7a6a
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for openfold
  Running setup.py clean for openfold
Failed to build openfold
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (openfold)


In [267]:
!pip uninstall -y fair-esm esm

Found existing installation: fair-esm 2.0.0
Uninstalling fair-esm-2.0.0:
  Successfully uninstalled fair-esm-2.0.0
Found existing installation: esm 3.4.0
Uninstalling esm-3.4.0:
  Successfully uninstalled esm-3.4.0


In [2]:
!pip install fair-esm[esmfold]

In [3]:
!pip install "dllogger @ git+https://github.com/NVIDIA/dllogger.git"

  Cloning https://github.com/NVIDIA/dllogger.git to /tmp/pip-install-qn6yaei9/dllogger_572687c88d4645ccbd033ed7f9fd8fcd
  Running command git clone --filter=blob:none --quiet https://github.com/NVIDIA/dllogger.git /tmp/pip-install-qn6yaei9/dllogger_572687c88d4645ccbd033ed7f9fd8fcd
  Resolved https://github.com/NVIDIA/dllogger.git to commit 0478734ff7be75adde8d160e04872664d1c62e5f
  Preparing metadata (setup.py) ... done


In [4]:
!pip install "openfold @ git+https://github.com/aqlaboratory/openfold.git@v1.0.1"

  Cloning https://github.com/aqlaboratory/openfold.git (to revision v1.0.1) to /tmp/pip-install-ag8f_qm6/openfold_8437cebf2e5f46d8b70719ba54be48f4
  Running command git clone --filter=blob:none --quiet https://github.com/aqlaboratory/openfold.git /tmp/pip-install-ag8f_qm6/openfold_8437cebf2e5f46d8b70719ba54be48f4
  Running command git checkout -q 42e71db7fa327e0810eb0e371abc9f82aa9b7a6a
  Resolved https://github.com/aqlaboratory/openfold.git to commit 42e71db7fa327e0810eb0e371abc9f82aa9b7a6a
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for openfold
  Running setup.py clean for openfold
Failed to build openfold
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (openfold)


In [10]:
import esm
import torch


esmfold_model = esm.pretrained.esmfold_v1()

esmfold_model = esmfold_model.eval().cuda()


print("ESMFold loaded successfully")

ModuleNotFoundError: No module named 'openfold'

!pip install biopython biotite

In [11]:
!pip install biopython biotite

Step 10.2 — Get AlphaFold structures

In [12]:
!wget https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/idmapping/by_organism/HUMAN_9606_idmapping.dat.gz

--2026-09-03 02:21:50--  https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/idmapping/by_organism/HUMAN_9606_idmapping.dat.gz
Resolving ftp.uniprot.org (ftp.uniprot.org)... 128.175.240.195
Connecting to ftp.uniprot.org (ftp.uniprot.org)|128.175.240.195|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 37842957 (36M) [application/x-gzip]
Saving to: ‘HUMAN_9606_idmapping.dat.gz’

HUMAN_9606_idmappin 100%[===================>]  36.09M  5.62MB/s    in 7.0s    

2026-09-03 02:21:58 (5.16 MB/s) - ‘HUMAN_9606_idmapping.dat.gz’ saved [37842957/37842957]



In [13]:
!gunzip HUMAN_9606_idmapping.dat.gz

In [14]:
import sys
import torch

print(sys.version)
print(torch.__version__)

3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
2.11.0+cu128


In [15]:
!wget https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/idmapping/by_organism/HUMAN_9606_idmapping.dat.gz

--2026-09-03 02:22:03--  https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/idmapping/by_organism/HUMAN_9606_idmapping.dat.gz
Resolving ftp.uniprot.org (ftp.uniprot.org)... 128.175.240.195
Connecting to ftp.uniprot.org (ftp.uniprot.org)|128.175.240.195|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 37842957 (36M) [application/x-gzip]
Saving to: ‘HUMAN_9606_idmapping.dat.gz’

HUMAN_9606_idmappin 100%[===================>]  36.09M  9.08MB/s    in 4.0s    

2026-09-03 02:22:08 (9.08 MB/s) - ‘HUMAN_9606_idmapping.dat.gz’ saved [37842957/37842957]



In [16]:
!gunzip HUMAN_9606_idmapping.dat.gz

gzip: HUMAN_9606_idmapping.dat already exists; do you wish to overwrite (y or n)? y


In [17]:
mapping_file="/content/HUMAN_9606_idmapping.dat"

with open(mapping_file) as f:
    for i in range(5):
        print(next(f))

P31946	UniProtKB-ID	1433B_HUMAN

P31946	Gene_Name	YWHAB

P31946	GI	78101741

P31946	GI	21328448

P31946	GI	377656702



In [18]:
ensembl_to_uniprot = {}

with open(mapping_file) as f:

    for line in f:

        parts=line.strip().split("\t")

        if len(parts)==3:

            uni_id, db, accession = parts

            if db=="Ensembl_PRO":

                ensembl_to_uniprot[accession]=uni_id


print(
    "Mappings:",
    len(ensembl_to_uniprot)
)

Mappings: 246047


In [19]:
import torch


embedding_path = "/content/MIP-FM/data/embeddings/human_esm2_8M_vectors.pt"


protein_embeddings = torch.load(
    embedding_path,
    map_location="cpu"
)


print(
    "Proteins loaded:",
    len(protein_embeddings)
)

Proteins loaded: 15816


In [20]:
test_ids = [
    x.split(".")[1]
    for x in list(protein_embeddings.keys())[:10]
]


for x in test_ids:

    print(
        x,
        "→",
        ensembl_to_uniprot.get(x)
    )

ENSP00000365953 → None
ENSP00000473392 → None
ENSP00000432512 → None
ENSP00000482566 → None
ENSP00000425719 → None
ENSP00000422858 → None
ENSP00000376349 → None
ENSP00000342570 → None
ENSP00000432386 → None
ENSP00000297540 → None


In [21]:
list(protein_embeddings.keys())[:10]

['9606.ENSP00000365953',
 '9606.ENSP00000473392',
 '9606.ENSP00000432512',
 '9606.ENSP00000482566',
 '9606.ENSP00000425719',
 '9606.ENSP00000422858',
 '9606.ENSP00000376349',
 '9606.ENSP00000342570',
 '9606.ENSP00000432386',
 '9606.ENSP00000297540']

In [22]:
test_ids = [
    x.split(".")[1]
    for x in list(protein_embeddings.keys())[:10]
]


for x in test_ids:

    print(
        x,
        "→",
        ensembl_to_uniprot.get(x)
    )

ENSP00000365953 → None
ENSP00000473392 → None
ENSP00000432512 → None
ENSP00000482566 → None
ENSP00000425719 → None
ENSP00000422858 → None
ENSP00000376349 → None
ENSP00000342570 → None
ENSP00000432386 → None
ENSP00000297540 → None


Step 10.5 — Build Ensembl → UniProt → AlphaFold pipeline

Step 1 — Reload embeddings (if needed)

In [23]:
import torch

protein_embeddings = torch.load(
    "/content/MIP-FM/data/embeddings/human_esm2_8M_vectors.pt",
    map_location="cpu"
)

print(len(protein_embeddings))

15816


In [24]:
test_ids = [
    x.split(".")[1]
    for x in list(protein_embeddings.keys())[:10]
]


for x in test_ids:
    print(
        x,
        "→",
        ensembl_to_uniprot.get(x)
    )

ENSP00000365953 → None
ENSP00000473392 → None
ENSP00000432512 → None
ENSP00000482566 → None
ENSP00000425719 → None
ENSP00000422858 → None
ENSP00000376349 → None
ENSP00000342570 → None
ENSP00000432386 → None
ENSP00000297540 → None


Step 3 — Create AlphaFold download function

In [25]:
import requests
import os


def download_alphafold_structure(uniprot_id, save_dir):

    os.makedirs(
        save_dir,
        exist_ok=True
    )


    url = (
        f"https://alphafold.ebi.ac.uk/files/"
        f"AF-{uniprot_id}-F1-model_v4.pdb"
    )


    output = (
        f"{save_dir}/"
        f"AF-{uniprot_id}.pdb"
    )


    r = requests.get(url)


    if r.status_code == 200:

        with open(output,"w") as f:
            f.write(r.text)

        return output

    else:

        return None

In [26]:
uniprot_id = "P04637"


structure_file = download_alphafold_structure(
    uniprot_id,
    "/content/MIP-FM/data/structures"
)


print(structure_file)

None


In [27]:
mapping_file="/content/HUMAN_9606_idmapping.dat"

with open(mapping_file) as f:
    for i in range(10):
        print(next(f))

P31946	UniProtKB-ID	1433B_HUMAN

P31946	Gene_Name	YWHAB

P31946	GI	78101741

P31946	GI	21328448

P31946	GI	377656702

P31946	GI	1034625756

P31946	GI	4507949

P31946	GI	67464628

P31946	GI	377656701

P31946	GI	1345590



In [28]:
ensembl_to_uniprot = {}

mapping_file="/content/HUMAN_9606_idmapping.dat"

with open(mapping_file) as f:

    for line in f:

        parts=line.strip().split("\t")

        if len(parts)==3:

            uniprot, db, identifier = parts

            if db == "Ensembl_PRO":

                ensembl_to_uniprot[identifier] = uniprot


print("Total mappings:", len(ensembl_to_uniprot))

Total mappings: 246047


In [29]:
test_ids = [
    x.split(".")[1]
    for x in list(protein_embeddings.keys())[:20]
]


for x in test_ids:
    print(
        x,
        "→",
        ensembl_to_uniprot.get(x)
    )

ENSP00000365953 → None
ENSP00000473392 → None
ENSP00000432512 → None
ENSP00000482566 → None
ENSP00000425719 → None
ENSP00000422858 → None
ENSP00000376349 → None
ENSP00000342570 → None
ENSP00000432386 → None
ENSP00000297540 → None
ENSP00000386756 → None
ENSP00000432713 → None
ENSP00000405975 → None
ENSP00000468235 → None
ENSP00000445233 → None
ENSP00000439397 → None
ENSP00000451835 → None
ENSP00000419325 → None
ENSP00000400874 → None
ENSP00000410007 → None


In [30]:
import requests
import os


def download_af(uniprot_id):

    url=f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb"

    r=requests.get(url)

    if r.status_code==200:

        path=f"/content/MIP-FM/data/structures/AF-{uniprot_id}.pdb"

        os.makedirs(
            "/content/MIP-FM/data/structures",
            exist_ok=True
        )

        open(path,"w").write(r.text)

        return path

    return None

In [31]:
download_af("P31946")

In [32]:
print("Total mappings:", len(ensembl_to_uniprot))

Total mappings: 246047


In [33]:
print("Total mappings:", len(ensembl_to_uniprot))

Total mappings: 246047


In [34]:
protein_ids = list(protein_embeddings.keys())


mapped_proteins = {}


for pid in protein_ids:

    ensembl_id = pid.split(".")[1]

    if ensembl_id in ensembl_to_uniprot:

        mapped_proteins[pid] = ensembl_to_uniprot[ensembl_id]


print(
    "Proteins with UniProt mapping:",
    len(mapped_proteins)
)

Proteins with UniProt mapping: 78


In [35]:
test_pid = list(mapped_proteins.keys())[0]

uniprot_id = mapped_proteins[test_pid]


print(test_pid)
print(uniprot_id)

9606.ENSP00000415516
A0A0A0MT54


In [36]:
import requests
import os


def download_alphafold(uniprot_id):

    url = (
        f"https://alphafold.ebi.ac.uk/files/"
        f"AF-{uniprot_id}-F1-model_v4.pdb"
    )


    response = requests.get(url)


    if response.status_code == 200:

        os.makedirs(
            "/content/MIP-FM/data/structures",
            exist_ok=True
        )

        path = (
            f"/content/MIP-FM/data/structures/"
            f"AF-{uniprot_id}.pdb"
        )


        with open(path,"w") as f:
            f.write(response.text)


        return path

    else:
        return None

In [37]:
structure_file = download_alphafold(
    uniprot_id
)


print(structure_file)

None


In [38]:
print(len(ensembl_to_uniprot))
list(ensembl_to_uniprot.items())[:5]
list(protein_embeddings.keys())[:5]

246047


['9606.ENSP00000365953',
 '9606.ENSP00000473392',
 '9606.ENSP00000432512',
 '9606.ENSP00000482566',
 '9606.ENSP00000425719']

In [39]:
pid = list(protein_embeddings.keys())[0]

print("Original:")
print(pid)


ensembl_id = pid.split(".")[1]

print("Converted:")
print(ensembl_id)


print("UniProt:")
print(ensembl_to_uniprot.get(ensembl_id))

Original:
9606.ENSP00000365953
Converted:
ENSP00000365953
UniProt:
None


In [40]:
mapped_proteins = {}

for pid in protein_embeddings.keys():

    ensembl_id = pid.split(".")[1]

    uni = ensembl_to_uniprot.get(ensembl_id)

    if uni:
        mapped_proteins[pid] = uni


print(
    "Total proteins:",
    len(protein_embeddings)
)

print(
    "Mapped proteins:",
    len(mapped_proteins)
)

Total proteins: 15816
Mapped proteins: 78


In [41]:
!wget https://stringdb-downloads.org/download/protein.aliases.v12.0/9606.protein.aliases.v12.0.txt.gz

--2026-09-03 02:24:41--  https://stringdb-downloads.org/download/protein.aliases.v12.0/9606.protein.aliases.v12.0.txt.gz
Resolving stringdb-downloads.org (stringdb-downloads.org)... 49.12.123.75
Connecting to stringdb-downloads.org (stringdb-downloads.org)|49.12.123.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 19777800 (19M) [application/octet-stream]
Saving to: ‘9606.protein.aliases.v12.0.txt.gz’

9606.protein.aliase 100%[===================>]  18.86M  10.2MB/s    in 1.9s    

2026-09-03 02:24:44 (10.2 MB/s) - ‘9606.protein.aliases.v12.0.txt.gz’ saved [19777800/19777800]



In [46]:
!gunzip 9606.protein.aliases.v12.0.txt.gz

In [43]:
!head -5 9606.protein.aliases.v12.0.txt

head: cannot open '9606.protein.aliases.v12.0.txt' for reading: No such file or directory


In [47]:
string_to_uniprot = {}

with open(
    "/content/9606.protein.aliases.v12.0.txt"
) as f:

    for line in f:

        if line.startswith("#"):
            continue

        parts=line.strip().split("\t")

        if len(parts)==3:

            string_id, alias, source = parts

            if source == "UniProt_AC":

                string_to_uniprot[string_id]=alias


print(
    "STRING-UniProt mappings:",
    len(string_to_uniprot)
)

STRING-UniProt mappings: 19399


In [48]:
test_ids = list(protein_embeddings.keys())[:10]


for pid in test_ids:

    print(
        pid,
        "→",
        string_to_uniprot.get(pid)
    )

9606.ENSP00000365953 → None
9606.ENSP00000473392 → None
9606.ENSP00000432512 → None
9606.ENSP00000482566 → None
9606.ENSP00000425719 → None
9606.ENSP00000422858 → None
9606.ENSP00000376349 → Q86SJ8
9606.ENSP00000342570 → Q6UXN2
9606.ENSP00000432386 → None
9606.ENSP00000297540 → Q9H8W1


In [49]:
string_mapped_proteins = {}

for pid in protein_embeddings.keys():

    uni = string_to_uniprot.get(pid)

    if uni:
        string_mapped_proteins[pid] = uni


print("Total proteins:", len(protein_embeddings))

print(
    "STRING → UniProt mapped:",
    len(string_mapped_proteins)
)

print(
    "Coverage:",
    len(string_mapped_proteins)/len(protein_embeddings)*100,
    "%"
)

Total proteins: 15816
STRING → UniProt mapped: 6146
Coverage: 38.859382903388976 %


In [50]:
import os

structure_dir="/content/MIP-FM/data/structures"

os.makedirs(
    structure_dir,
    exist_ok=True
)

print(structure_dir)

/content/MIP-FM/data/structures


In [51]:
import requests
import os


def download_alphafold(uniprot_id):

    url = (
        f"https://alphafold.ebi.ac.uk/files/"
        f"AF-{uniprot_id}-F1-model_v4.pdb"
    )


    path = (
        f"{structure_dir}/"
        f"AF-{uniprot_id}.pdb"
    )


    if os.path.exists(path):
        return path


    response = requests.get(
        url,
        timeout=30
    )


    if response.status_code == 200:

        with open(path,"w") as f:
            f.write(response.text)

        return path


    return None

In [52]:
from tqdm import tqdm


test_uniprots = list(
    string_mapped_proteins.values()
)[:10]


for uni in tqdm(test_uniprots):

    result = download_alphafold(uni)

    print(
        uni,
        "→",
        result
    )

 10%|█         | 1/10 [00:00<00:03,  2.77it/s]

Q86SJ8 → None


 20%|██        | 2/10 [00:00<00:02,  2.80it/s]

Q6UXN2 → None


 30%|███       | 3/10 [00:01<00:03,  2.02it/s]

...

P18146 → None


 90%|█████████ | 9/10 [00:03<00:00,  2.66it/s]

Q9HB07 → None


100%|██████████| 10/10 [00:04<00:00,  2.42it/s]

Q9UFC2 → None


In [53]:
import requests


uniprot_id="Q8WXL4"


url=f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"


r=requests.get(url)


print(r.status_code)
print(r.text[:500])

404
{}


In [54]:
import os

files=os.listdir(
    "/content/MIP-FM/data/structures"
)


print(
    "Structures downloaded:",
    len(files)
)

print(files[:5])

Structures downloaded: 0
[]


In [55]:
import requests

uniprot_id = "Q8WXL4"

url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"

r = requests.get(url)

print("Status:", r.status_code)
print(r.text[:500])

Status: 404
{}


In [56]:
import requests

uniprot_id="P04637"

url=f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"

r=requests.get(url)

print("Status:", r.status_code)
print(r.text[:500])

Status: 200
[{"toolUsed":"AlphaFold Monomer v2.0 pipeline","providerId":"GDM","entityType":"protein","isUniProt":true,"modelEntityId":"AF-P04637-F1","modelCreatedDate":"2025-08-01T00:00:00Z","sequenceVersionDate":"2009-11-24T00:00:00Z","globalMetricValue":75.06,"fractionPlddtVeryLow":0.298,"fractionPlddtLow":0.104,"fractionPlddtConfident":0.071,"fractionPlddtVeryHigh":0.527,"latestVersion":6,"allVersions":[1,2,3,4,5,6],"sequence":"MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGPDEAPRMPEAAPPVAPAP


In [57]:
mapped_uniprots = list(
    string_mapped_proteins.values()
)

print(
    "Mapped UniProt proteins:",
    len(mapped_uniprots)
)

Mapped UniProt proteins: 6146


In [58]:
import requests
from tqdm import tqdm


def alphafold_available(uniprot_id):

    url = (
        f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
    )

    try:

        r = requests.get(
            url,
            timeout=10
        )

        if r.status_code == 200:

            return True

        return False

    except:

        return False

In [59]:
available_test=[]


for uni in tqdm(mapped_uniprots[:50]):

    if alphafold_available(uni):

        available_test.append(uni)


print(
    "Available structures:",
    len(available_test)
)

100%|██████████| 50/50 [00:05<00:00,  8.53it/s]

Available structures: 29


In [60]:
available_structures=[]


for uni in tqdm(mapped_uniprots):

    if alphafold_available(uni):

        available_structures.append(uni)


print(
    len(available_structures)
)

100%|██████████| 6146/6146 [12:01<00:00,  8.52it/s]

2955


In [61]:
import requests


def get_alphafold_pdb_url(uniprot_id):

    url=f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"

    data=requests.get(url).json()

    return data[0]["pdbUrl"]

In [62]:
get_alphafold_pdb_url("P04637")

'https://alphafold.ebi.ac.uk/files/AF-P04637-F1-model_v6.pdb'

In [63]:
import requests
import os


structure_dir="/content/MIP-FM/data/structures"

os.makedirs(
    structure_dir,
    exist_ok=True
)


def download_alphafold(uniprot_id):

    # Get metadata
    api_url = (
        f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
    )

    response = requests.get(api_url)

    if response.status_code != 200:
        return None


    data = response.json()


    # Current PDB URL
    pdb_url = data[0]["pdbUrl"]


    pdb_response = requests.get(
        pdb_url
    )


    if pdb_response.status_code == 200:

        path = (
            f"{structure_dir}/"
            f"AF-{uniprot_id}.pdb"
        )

        with open(path,"wb") as f:
            f.write(
                pdb_response.content
            )

        return path


    return None

In [64]:
path = download_alphafold(
    "P04637"
)

print(path)

/content/MIP-FM/data/structures/AF-P04637.pdb


In [65]:
test_uniprots = list(
    string_mapped_proteins.values()
)[:20]


downloaded=[]


for uni in test_uniprots:

    result = download_alphafold(
        uni
    )

    if result:
        downloaded.append(uni)


print(
    "Downloaded:",
    len(downloaded)
)

print(downloaded)

Downloaded: 12
['Q6UXN2', 'Q96I99', 'Q6PJ61', 'Q8TCP9', 'P18146', 'Q9HB07', 'Q8NE28', 'P0CG40', 'Q9P2Z0', 'Q9NX57', 'Q6PXP3', 'Q15365']


In [66]:
!pip install biopython biotite

Step 13.2 — Load one PDB file

In [67]:
import os

structure_files = os.listdir(
    "/content/MIP-FM/data/structures"
)

print(structure_files[:5])

['AF-Q96I99.pdb', 'AF-Q9HB07.pdb', 'AF-Q9P2Z0.pdb', 'AF-Q8TCP9.pdb', 'AF-P0CG40.pdb']


In [79]:
from Bio.PDB import PDBParser


pdb_path="/content/MIP-FM/data/structures/AF-Q9P2Z0.pdb"


parser = PDBParser(
    QUIET=True
)


structure = parser.get_structure(
    "protein",
    pdb_path
)


print(structure)

<Structure id=protein>


Step 13.4 — Extract alpha-carbon coordinates

In [80]:
import numpy as np


coords=[]


for model in structure:

    for chain in model:

        for residue in chain:

            if "CA" in residue:

                ca = residue["CA"]

                coords.append(
                    ca.coord
                )


coords=np.array(coords)


print(
    "Residues:",
    len(coords)
)

print(
    "Coordinate shape:",
    coords.shape
)

Residues: 257
Coordinate shape: (257, 3)


In [81]:
from scipy.spatial.distance import cdist


distance_matrix = cdist(
    coords,
    coords
)


contact_map = (
    distance_matrix < 8.0
)


print(contact_map.shape)

(257, 257)


Step 14 — Build the Structure Graph Encoder

Residue graph
        |
        ↓
Graph Neural Network
        |
        ↓
128-dimensional structure embedding

In [75]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.9 MB/s eta 0:00:00


Step 14.2 — Create residue node features

First we need residue information.

For each residue we will encode:

Amino acid identity
3D coordinate
pLDDT confidence (AlphaFold B-factor)

In [82]:
import torch
import numpy as np
from Bio.PDB import PDBParser


amino_acids = {
    "ALA":0,"ARG":1,"ASN":2,"ASP":3,
    "CYS":4,"GLN":5,"GLU":6,"GLY":7,
    "HIS":8,"ILE":9,"LEU":10,"LYS":11,
    "MET":12,"PHE":13,"PRO":14,"SER":15,
    "THR":16,"TRP":17,"TYR":18,"VAL":19
}


def extract_structure_features(pdb_file):

    parser=PDBParser(QUIET=True)

    structure=parser.get_structure(
        "protein",
        pdb_file
    )


    features=[]

    coords=[]


    for model in structure:

        for chain in model:

            for residue in chain:

                if "CA" in residue:

                    aa=residue.resname

                    aa_id=amino_acids.get(
                        aa,
                        20
                    )

                    ca=residue["CA"]

                    coord=ca.coord

                    plddt=ca.bfactor


                    node=[
                        aa_id,
                        coord[0],
                        coord[1],
                        coord[2],
                        plddt
                    ]

                    features.append(node)

                    coords.append(coord)


    return (
        np.array(features),
        np.array(coords)
    )

Step 14.3 — Test on your downloaded structure

In [83]:
pdb_file="/content/MIP-FM/data/structures/AF-Q9P2Z0.pdb"


node_features, coords = extract_structure_features(
    pdb_file
)


print(node_features.shape)
print(coords.shape)

(257, 5)
(257, 3)


Step 14.4 — Convert contact map into graph edges

In [84]:
from scipy.spatial.distance import cdist


distance_matrix = cdist(
    coords,
    coords
)


edges=np.where(
    distance_matrix < 8.0
)


edge_index=torch.tensor(
    np.vstack(edges),
    dtype=torch.long
)


print(edge_index.shape)

torch.Size([2, 1719])


Residue graph

Nodes:
373 × 5 features

Edges:
Residue contacts

        ↓

Graph Neural Network

        ↓

128-dimensional structure embedding

Step 15 — Create GNN Structure Encoder

In [85]:
import torch
import torch.nn as nn

from torch_geometric.nn import GCNConv
from torch_geometric.nn import global_mean_pool

Step 15.2 — Define Structure GNN

In [86]:
class StructureEncoder(nn.Module):

    def __init__(
        self,
        input_dim=5,
        hidden_dim=64,
        output_dim=128
    ):

        super().__init__()

        self.conv1 = GCNConv(
            input_dim,
            hidden_dim
        )

        self.conv2 = GCNConv(
            hidden_dim,
            hidden_dim
        )

        self.fc = nn.Linear(
            hidden_dim,
            output_dim
        )


    def forward(
        self,
        x,
        edge_index
    ):

        x = self.conv1(
            x,
            edge_index
        )

        x = torch.relu(x)


        x = self.conv2(
            x,
            edge_index
        )

        x = torch.relu(x)


        x = torch.mean(
            x,
            dim=0
        )


        x = self.fc(x)

        return x

Step 15.3 — Convert your protein graph to tensors

In [87]:
node_features
coords
edge_index

tensor([[  0,   0,   0,  ..., 256, 256, 256],
        [  0,   1,   2,  ..., 254, 255, 256]])

In [88]:
x = torch.tensor(
    node_features,
    dtype=torch.float
)


print(x.shape)

torch.Size([257, 5])


In [89]:
x = x.cuda()

edge_index = edge_index.cuda()

In [90]:
structure_encoder = StructureEncoder()

structure_encoder = structure_encoder.cuda()

print("Structure encoder ready")

Structure encoder ready


In [91]:
node_features
edge_index

x = torch.tensor(
    node_features,
    dtype=torch.float
).cuda()


edge_index = edge_index.cuda()


print(x.shape)
print(edge_index.shape)

torch.Size([257, 5])
torch.Size([2, 1719])


In [92]:
with torch.no_grad():

    structure_embedding = structure_encoder(
        x,
        edge_index
    )


print(structure_embedding.shape)

torch.Size([128])


Step 15.4 — Generate first structure embedding

In [93]:
import torch
import torch.nn as nn

from torch_geometric.nn import GCNConv
from torch_geometric.nn import global_mean_pool

Step 16 — Create multimodal protein representation

ESM-2 embedding
      |
      |
    320 dims


Structure GNN embedding
      |
      |
    128 dims


====================

Combined representation

448 dims

In [94]:
protein_id = "9606.ENSP00000331600"

In [95]:
esm_embedding = protein_embeddings[protein_id]

print(esm_embedding.shape)

torch.Size([320])


In [96]:
combined_embedding = torch.cat(
    [
        esm_embedding.cpu(),
        structure_embedding.cpu()
    ]
)


print(combined_embedding.shape)

torch.Size([448])


Step 17 — Generate structure embeddings for all downloaded proteins

In [97]:
import os

structure_files = os.listdir(
    "/content/MIP-FM/data/structures"
)

print(len(structure_files))
print(structure_files[:5])

13
['AF-Q96I99.pdb', 'AF-Q9HB07.pdb', 'AF-Q9P2Z0.pdb', 'AF-Q8TCP9.pdb', 'AF-P0CG40.pdb']


In [98]:
def get_structure_embedding(pdb_file):

    node_features, coords = extract_structure_features(
        pdb_file
    )


    from scipy.spatial.distance import cdist

    distance_matrix = cdist(
        coords,
        coords
    )


    edges = np.where(
        distance_matrix < 8.0
    )


    edge_index = torch.tensor(
        np.vstack(edges),
        dtype=torch.long
    )


    x = torch.tensor(
        node_features,
        dtype=torch.float
    ).cuda()


    edge_index=edge_index.cuda()


    with torch.no_grad():

        emb = structure_encoder(
            x,
            edge_index
        )


    return emb.cpu()

In [99]:
structure_embeddings = {}


for file in structure_files:

    uni_id = file.replace(
        "AF-",
        ""
    ).replace(
        ".pdb",
        ""
    )


    path = (
        "/content/MIP-FM/data/structures/"
        + file
    )


    structure_embeddings[uni_id] = (
        get_structure_embedding(path)
    )


print(
    len(structure_embeddings)
)

13


11 AlphaFold structures
        |
        ↓
Structure GNN
        |
        ↓
11 × 128-dimensional vectors

Step 18 — Create the multimodal embedding dictionary

UniProt ID
      |
      |
ESM-2 embedding (320)
      +
Structure embedding (128)

      ↓

448-dimensional vector

In [100]:
uniprot_to_string = {}

for string_id, uni_id in string_to_uniprot.items():

    uniprot_to_string[uni_id] = string_id


print(len(uniprot_to_string))

19362


Step 18.2 — Combine embeddings

In [101]:
multimodal_embeddings = {}


for uni_id, struct_emb in structure_embeddings.items():

    if uni_id in uniprot_to_string:

        string_id = uniprot_to_string[uni_id]


        if string_id in protein_embeddings:

            esm_emb = protein_embeddings[string_id]


            combined = torch.cat(
                [
                    esm_emb.cpu(),
                    struct_emb.cpu()
                ]
            )


            multimodal_embeddings[string_id] = combined



print(
    "Multimodal proteins:",
    len(multimodal_embeddings)
)

Multimodal proteins: 12


In [102]:
pid = list(multimodal_embeddings.keys())[0]


print(pid)

print(
    multimodal_embeddings[pid].shape
)

9606.ENSP00000419325
torch.Size([448])


Step 19 — Build the Structure-Aware PPI Dataset

Step 19.2 — Filter multimodal pairs

In [103]:
import os

for root, dirs, files in os.walk("/content/MIP-FM"):
    for f in files:
        if f.endswith((".csv",".tsv",".txt")):
            print(os.path.join(root,f))

/content/MIP-FM/data/sequences/ecoli_test.tsv
/content/MIP-FM/data/sequences/yeast_test.tsv
/content/MIP-FM/data/sequences/fly_test.tsv
/content/MIP-FM/data/sequences/worm_test.tsv
/content/MIP-FM/data/sequences/mouse_test.tsv


In [104]:
import pandas as pd

ppi_df = pd.read_csv(
    "/content/human_train.tsv",
    sep="\t"
)

print(ppi_df.shape)

ppi_df.head()

(421791, 3)


,9606.ENSP00000409077,9606.ENSP00000470819,1
0,9606.ENSP00000263904,9606.ENSP00000472680,1
1,9606.ENSP00000364459,9606.ENSP00000360117,1
2,9606.ENSP00000422403,9606.ENSP00000400591,1
3,9606.ENSP00000388332,9606.ENSP00000346080,1
4,9606.ENSP00000469581,9606.ENSP00000386920,1


In [105]:
import pandas as pd

file_path="/content/human_train.tsv"

ppi_df = pd.read_csv(
    file_path,
    sep="\t",
    header=None
)

ppi_df.columns=[
    "protein1",
    "protein2",
    "label"
]


print(ppi_df.shape)

ppi_df.head()

(421792, 3)


,protein1,protein2,label
0,9606.ENSP00000409077,9606.ENSP00000470819,1
1,9606.ENSP00000263904,9606.ENSP00000472680,1
2,9606.ENSP00000364459,9606.ENSP00000360117,1
3,9606.ENSP00000422403,9606.ENSP00000400591,1
4,9606.ENSP00000388332,9606.ENSP00000346080,1


In [106]:
multimodal_pairs = ppi_df[
    (ppi_df["protein1"].isin(multimodal_embeddings.keys())) &
    (ppi_df["protein2"].isin(multimodal_embeddings.keys()))
]


print(
    "Original pairs:",
    len(ppi_df)
)

print(
    "Multimodal pairs:",
    len(multimodal_pairs)
)

Original pairs: 421792
Multimodal pairs: 0


In [ ]:
from tqdm import tqdm
import requests


available_uniprots = []


for uni in tqdm(string_mapped_proteins.values()):

    try:

        url = f"https://alphafold.ebi.ac.uk/api/prediction/{uni}"

        r = requests.get(
            url,
            timeout=10
        )

        if r.status_code == 200:
            available_uniprots.append(uni)

    except:
        pass


print(
    "Available AlphaFold structures:",
    len(available_uniprots)
)

 50%|████▉     | 3060/6146 [05:58<06:08,  8.39it/s]

In [ ]:
from tqdm import tqdm


downloaded_structures = []


for uni in tqdm(available_uniprots[:500]):

    path = download_alphafold(uni)

    if path:
        downloaded_structures.append(uni)


print(
    "Downloaded:",
    len(downloaded_structures)
)

structure_embeddings = {}


for uni in tqdm(downloaded_structures):

    pdb_file = (
        f"/content/MIP-FM/data/structures/"
        f"AF-{uni}.pdb"
    )

    try:

        emb = get_structure_embedding(
            pdb_file
        )

        structure_embeddings[uni]=emb

    except Exception as e:
        print(
            "Failed:",
            uni,
            e
        )


print(
    "Structure embeddings:",
    len(structure_embeddings)
)


structure_embeddings = {}


for uni in tqdm(downloaded_structures):

    pdb_file = (
        f"/content/MIP-FM/data/structures/"
        f"AF-{uni}.pdb"
    )

    try:

        emb = get_structure_embedding(
            pdb_file
        )

        structure_embeddings[uni]=emb

    except Exception as e:
        print(
            "Failed:",
            uni,
            e
        )


print(
    "Structure embeddings:",
    len(structure_embeddings)
)

In [ ]:
structure_embeddings = {}


for uni in tqdm(downloaded_structures):

    pdb_file = (
        f"/content/MIP-FM/data/structures/"
        f"AF-{uni}.pdb"
    )

    try:

        emb = get_structure_embedding(
            pdb_file
        )

        structure_embeddings[uni]=emb

    except Exception as e:
        print(
            "Failed:",
            uni,
            e
        )


print(
    "Structure embeddings:",
    len(structure_embeddings)
)

AlphaFold structures downloaded: 500

Structure embeddings generated: 500

Protein sequence
        |
        ↓
ESM-2
        |
        ↓
320-dimensional vector


Protein structure
        |
        ↓
AlphaFold PDB
        |
        ↓
Structure GNN
        |
        ↓
128-dimensional vector


        ↓

448-dimensional protein representation

In [ ]:
multimodal_embeddings = {}


for uni_id, struct_emb in structure_embeddings.items():

    # UniProt → STRING
    if uni_id in uniprot_to_string:

        string_id = uniprot_to_string[uni_id]


        # STRING → ESM-2
        if string_id in protein_embeddings:

            esm_emb = protein_embeddings[string_id]


            combined = torch.cat(
                [
                    esm_emb.cpu(),
                    struct_emb.cpu()
                ]
            )


            multimodal_embeddings[string_id] = combined



print(
    "Multimodal proteins:",
    len(multimodal_embeddings)
)

In [ ]:
ppi_df

In [ ]:
multimodal_pairs = ppi_df[
    (ppi_df["protein1"].isin(multimodal_embeddings.keys())) &
    (ppi_df["protein2"].isin(multimodal_embeddings.keys()))
]


print(
    "Original pairs:",
    len(ppi_df)
)

print(
    "Structure-aware pairs:",
    len(multimodal_pairs)
)

In [ ]:
import torch


X = []
y = []


for _, row in multimodal_pairs.iterrows():

    p1 = row["protein1"]
    p2 = row["protein2"]

    emb1 = multimodal_embeddings[p1]
    emb2 = multimodal_embeddings[p2]


    pair_embedding = torch.cat(
        [
            emb1,
            emb2
        ]
    )


    X.append(pair_embedding)
    y.append(row["label"])


X = torch.stack(X)

y = torch.tensor(
    y,
    dtype=torch.float
)


print(X.shape)
print(y.shape)

We have reached the point where we can train the structure-aware PPI model.

Step 24 — Train the multimodal classifier

Step 24 — Split multimodal dataset

In [ ]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print(X_train.shape)
print(X_test.shape)

Step 25 — Define multimodal PPI classifier

In [ ]:
import torch.nn as nn


class MultiModalPPINetwork(nn.Module):

    def __init__(self):

        super().__init__()

        self.model = nn.Sequential(

            nn.Linear(896,512),

            nn.ReLU(),

            nn.Dropout(0.3),


            nn.Linear(512,256),

            nn.ReLU(),

            nn.Dropout(0.3),


            nn.Linear(256,64),

            nn.ReLU(),


            nn.Linear(64,1),

            nn.Sigmoid()
        )


    def forward(self,x):

        return self.model(x)

Step 26 — Train model

In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"


model = MultiModalPPINetwork().to(device)


criterion = nn.BCELoss()


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)


X_train = X_train.to(device)
X_test = X_test.to(device)

y_train = y_train.to(device)
y_test = y_test.to(device)

In [ ]:
from tqdm import tqdm


epochs=50


for epoch in range(epochs):

    model.train()


    optimizer.zero_grad()


    pred = model(
        X_train.float()
    ).squeeze()


    loss = criterion(
        pred,
        y_train.float()
    )


    loss.backward()

    optimizer.step()


    if (epoch+1)%10==0:

        print(
            "Epoch:",
            epoch+1,
            "Loss:",
            loss.item()
        )

Step 27 — Evaluate

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    matthews_corrcoef
)


model.eval()


with torch.no_grad():

    probs = model(
        X_test.float()
    ).squeeze()


preds = (
    probs > 0.5
).cpu().numpy()


y_true = y_test.cpu().numpy()


print(
    "Accuracy:",
    accuracy_score(
        y_true,
        preds
    )
)


print(
    "AUROC:",
    roc_auc_score(
        y_true,
        probs.cpu().numpy()
    )
)


print(
    "AUPR:",
    average_precision_score(
        y_true,
        probs.cpu().numpy()
    )
)


print(
    "F1:",
    f1_score(
        y_true,
        preds
    )
)


print(
    "MCC:",
    matthews_corrcoef(
        y_true,
        preds
    )
)

In [ ]:
import torch
import os


save_dir="/content/MIP-FM/data/multimodal"

os.makedirs(
    save_dir,
    exist_ok=True
)


torch.save(
    multimodal_embeddings,
    f"{save_dir}/protein_multimodal_embeddings.pt"
)


print("Saved")

Compelet Architecture

                    Protein A
                       |
        --------------------------------
        |                              |
        ↓                              ↓

  Amino Acid Sequence             AlphaFold Structure
        |                              |
        |                              |
        ↓                              ↓

     ESM-2 Encoder              Structure Graph Builder
        |                              |
        |                              |
        ↓                              ↓

 Sequence Embedding             Residue Contact Graph

     320 dimensions              Nodes + Edges


                                      |
                                      ↓

                              Structure GNN Encoder

                                      |
                                      ↓

                           Structure Embedding

                              128 dimensions


        |                              |
        |                              |
        -------------------------------
                       |
                       ↓

            Multimodal Protein Representation

                 320 + 128 = 448 dimensions


                       |
                       |
          ---------------------------------
          |                               |
          ↓                               ↓

       Protein A                      Protein B

        448-d                         448-d


          |                               |
          ---------------------------------

                       |
                       ↓

              Pairwise Feature Fusion

                 448 + 448

                 896 dimensions


                       |
                       ↓

              PPI Classification Network


          Linear(896 → 128)

                 ↓

              ReLU

                 ↓

             Dropout(0.5)

                 ↓

          Linear(128 → 32)

                 ↓

              ReLU

                 ↓

          Linear(32 → 1)

                 ↓

          Sigmoid / Probability


                       |
                       ↓

             Interaction Probability

             0 = Non-interacting
             1 = Interacting

Current Pipeline

DATA
 |
 |---- Human PPI dataset          ✅
 |
 |---- Protein sequences          ✅
 |
 ↓

SEQUENCE MODEL
 |
 |---- ESM-2 embeddings           ✅
 |
 |---- Baseline PPI classifier    ✅
 |
 ↓

STRUCTURE MODEL
 |
 |---- AlphaFold download         ✅
 |
 |---- PDB processing             ✅
 |
 |---- Residue graph              ✅
 |
 |---- GNN encoder                ✅
 |
 ↓

MULTIMODAL MODEL
 |
 |---- 448-d protein vector       ✅
 |
 |---- 896-d PPI vector           ✅
 |
 |---- Fusion classifier          ⚠️ needs scale
 |
 ↓

FINAL PAPER EXPERIMENT
 |
 |---- 5000+ structures            ❌
 |
 |---- Large multimodal training   ❌
 |
 |---- Ablation studies            ❌
 |
 |---- Final comparison            ❌

In [ ]:
import requests


def ensembl_lookup(protein_id):

    url = (
        "https://rest.ensembl.org/"
        f"lookup/id/{protein_id}"
    )

    headers={
        "Content-Type":"application/json"
    }


    response=requests.get(
        url,
        headers=headers
    )


    return response.json()



result = ensembl_lookup(
    "ENSP00000000233"
)


print(result)

In [ ]:
print(protein_ids_test[:5])

In [ ]:
mapping_result = map_ensembl_to_uniprot(
    ensembl_ids
)


print(mapping_result)

In [ ]:
import os

os.makedirs(
    "/content/MIP-FM/data/embeddings",
    exist_ok=True
)


torch.save(
    protein_embeddings,
    "/content/MIP-FM/data/embeddings/human_100_protein_vectors.pt"
)


print("Saved")

Step 3: Build the ESM-2 Embedding Pipeline.

In [ ]:
import esm
import torch


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print("Device:", device)


model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()


model = model.to(device)

model.eval()


batch_converter = alphabet.get_batch_converter()


print("ESM-2 loaded")
def get_esm_embedding(sequence):

    data = [
        ("protein", sequence)
    ]


    labels, strs, tokens = batch_converter(data)


    tokens = tokens.to(device)


    with torch.no_grad():

        results = model(
            tokens,
            repr_layers=[6],
            return_contacts=False
        )


    embedding = results["representations"][6]


    # remove special tokens (<cls>, <eos>)
    embedding = embedding[:,1:-1,:]


    return embedding.cpu()

In [ ]:
sequence = "MGLTVSALFSRIFGKKQMRILMVGLDAAGKTTILYKLKLGEIVTTIPTIG"


embedding = get_esm_embedding(sequence)


print(
    "Embedding shape:",
    embedding.shape
)

1 protein

52 amino acids

320-dimensional ESM representation

Step 3.5 — Connect With FASTA File

In [ ]:
from Bio import SeqIO


fasta_path="/content/human.fasta"


sequence_dict={}


for record in SeqIO.parse(
    fasta_path,
    "fasta"
):

    sequence_dict[record.id]=str(record.seq)



print(
    "Proteins loaded:",
    len(sequence_dict)
)

In [ ]:
import os

os.makedirs(
    "/content/MIP-FM/data/embeddings",
    exist_ok=True
)

Step 3.6 — Create Embedding Dictionary

In [ ]:
import torch
from tqdm import tqdm


embedding_dict={}


for protein_id, sequence in tqdm(sequence_dict.items()):

    embedding_dict[protein_id] = get_esm_embedding(sequence)



torch.save(
    embedding_dict,
    "/content/MIP-FM/data/embeddings/human_esm_embeddings.pt"
)


print("Embeddings saved")